# 08 — VLM Tagging Bake-Off

**Goal:** Find the cheapest vision-language model that produces the same structured garment tags as `gemini-2.5-flash` — the current model in `src/ai/categorize.ts`.

**Pipeline:**
```
5 fixed garment images
  → each model asked for the same JSON schema
  → parse rate (% of responses that produce clean JSON)
  → field-level agreement vs Gemini baseline
  → latency + actual billed cost table
  → decision: which model to swap into categorize.ts
```


In [3]:
%matplotlib inline
import os, sys, json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from PIL import Image
from dotenv import load_dotenv

load_dotenv(Path("..") / ".env")
sys.path.insert(0, str(Path("..")))

from lib.providers import GeminiProvider, OpenRouterProvider, all_providers, print_availability
from lib.parse import parse_gemini_json

OUT_DIR = Path("../outputs/08")
OUT_DIR.mkdir(parents=True, exist_ok=True)

providers = all_providers()
print("Provider status:")
print_availability(providers)


Provider status:
  gemini       ✓ ready
  openrouter   ✓ ready
  fal          ✓ ready


In [4]:
# ── Load 5 fixed garment images ──────────────────────────────────────────────
# Priority: inputs/worn/ → ~/.closet/images/items/
import glob
from lib.images import resize_image

def _load_images(n=5):
    worn = sorted((Path("../inputs/worn")).glob("*.jpg"))
    worn += sorted((Path("../inputs/worn")).glob("*.jpeg"))
    worn += sorted((Path("../inputs/worn")).glob("*.png"))
    worn = [p for p in worn if not p.name.startswith(".")]
    if len(worn) >= n:
        return worn[:n]
    db = sorted(glob.glob(os.path.expanduser("~/.closet/images/items/*.jpg")))
    paths = list(worn) + [Path(p) for p in db]
    return paths[:n]

IMAGE_PATHS = _load_images()
IMAGES = [resize_image(Image.open(p)) for p in IMAGE_PATHS]
print(f"Loaded {len(IMAGES)} image(s):")
for p in IMAGE_PATHS:
    print(f"  {p}")

cols = len(IMAGES)
fig, axes = plt.subplots(1, cols, figsize=(cols * 3, 4))
if cols == 1: axes = [axes]
for ax, img, p in zip(axes, IMAGES, IMAGE_PATHS):
    ax.imshow(img); ax.set_title(Path(p).name[:20], fontsize=7); ax.axis("off")
plt.suptitle("Input images", fontsize=11)
plt.tight_layout(); plt.show()


Loaded 5 image(s):
  ~/.closet/images/items/0VJj34XK0fWoDcNLtv7lD.jpg
  ~/.closet/images/items/0p3JdJ7b6QApb0AlNxC-T.jpg
  ~/.closet/images/items/1R4wawuERj1NcMseQf0KV.jpg
  ~/.closet/images/items/1uMpVs-0To82x1Pwfg11r.jpg
  ~/.closet/images/items/2iVS6sUrKA5jNuNMF3_Ij.jpg


[image removed from the public copy: third-party or personal photos]


In [5]:
# ── Tagging prompt + schema ───────────────────────────────────────────────────
SCHEMA = {
    "category": "tops|bottoms|shoes|outerwear|accessories|dresses|activewear|underwear",
    "subcategory": "specific type",
    "primary_color": "single color name",
    "colors": ["list of colors"],
    "material": "fabric type or null",
    "season": ["spring", "summer", "fall", "winter"],
    "ai_description": "1-sentence style description",
    "suggested_tags": ["casual", "formal", "streetwear", "etc"],
}

PROMPT = f"""Analyze this clothing item image.
Return ONLY valid JSON — no markdown, no explanation, no extra text.
Use exactly this schema:
{json.dumps(SCHEMA, indent=2)}"""

print("Prompt length:", len(PROMPT), "chars")
print("\nSchema keys:", list(SCHEMA.keys()))


Prompt length: 575 chars

Schema keys: ['category', 'subcategory', 'primary_color', 'colors', 'material', 'season', 'ai_description', 'suggested_tags']


In [6]:
# ── Run all models on all images ─────────────────────────────────────────────
# Results stored as: results[model_id][image_idx] = TextResult
from lib.providers import GeminiProvider, OpenRouterProvider

gemini   = providers["gemini"]
openrouter = providers["openrouter"]

MODELS = [
    ("gemini",      "gemini-2.5-flash"),
    ("openrouter",  "qwen/qwen3-vl-8b-instruct"),
    ("openrouter",  "qwen/qwen3-vl-30b-a3b-instruct"),
    ("openrouter",  "qwen/qwen2.5-vl-72b-instruct"),
    ("openrouter",  "meta-llama/llama-3.2-90b-vision-instruct"),
    ("openrouter",  "mistralai/pixtral-large-2411"),
]

results = {}  # model_id → list[TextResult]

for provider_name, model_id in MODELS:
    p = providers[provider_name]
    if not p.available:
        print(f"  SKIP {model_id} — {provider_name} key missing")
        continue
    print(f"\n[{model_id}]")
    model_results = []
    for i, img in enumerate(IMAGES):
        r = p.chat_vision(model_id, img, PROMPT)
        tag = "✓" if r.ok else "✗"
        cost_str = f"${r.cost_usd:.6f}{'*' if r.cost_is_actual else '~'}"
        print(f"  img{i}: {tag} {r.latency_s:.1f}s {cost_str}  {'err: '+r.error[:60] if r.error else ''}")
        model_results.append(r)
    results[model_id] = model_results

print("\nDone. * = actual billed cost, ~ = estimate")



[gemini-2.5-flash]
  img0: ✓ 6.1s $0.000540~  
  img1: ✓ 4.6s $0.000550~  
  img2: ✓ 3.6s $0.000585~  
  img3: ✓ 2.9s $0.000595~  
  img4: ✓ 3.0s $0.000552~  

[qwen/qwen3-vl-8b-instruct]
  img0: ✗ 0.5s $0.000000~  err: AuthenticationError: Error code: 401 - {'error': {'message':
  img1: ✗ 0.1s $0.000000~  err: AuthenticationError: Error code: 401 - {'error': {'message':
  img2: ✗ 0.2s $0.000000~  err: AuthenticationError: Error code: 401 - {'error': {'message':
  img3: ✗ 0.0s $0.000000~  err: AuthenticationError: Error code: 401 - {'error': {'message':
  img4: ✗ 0.1s $0.000000~  err: AuthenticationError: Error code: 401 - {'error': {'message':

[qwen/qwen3-vl-30b-a3b-instruct]
  img0: ✗ 0.0s $0.000000~  err: AuthenticationError: Error code: 401 - {'error': {'message':
  img1: ✗ 0.1s $0.000000~  err: AuthenticationError: Error code: 401 - {'error': {'message':
  img2: ✗ 0.1s $0.000000~  err: AuthenticationError: Error code: 401 - {'error': {'message':
  img3: ✗ 0.1s $0.000000~  err: A

In [7]:
# ── Parse results + compute parse rate ───────────────────────────────────────
import re

def try_parse(text: str) -> dict | None:
    """Parse JSON from model response — handles markdown fences."""
    if not text:
        return None
    try:
        return json.loads(text)
    except Exception:
        pass
    try:
        return parse_gemini_json(text)
    except Exception:
        pass
    # Last resort: strip code fences manually
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.MULTILINE)
    try:
        return json.loads(cleaned)
    except Exception:
        return None

parsed = {}  # model_id → list[dict | None]
for model_id, model_results in results.items():
    parsed[model_id] = [try_parse(r.text) if r.ok else None for r in model_results]

# Parse rate table
print(f"{'Model':<52}  {'Parse rate':>12}  {'Fields present (avg)':>22}")
print("─" * 92)
schema_keys = list(SCHEMA.keys())
for model_id, parses in parsed.items():
    valid = [p for p in parses if p is not None]
    rate = len(valid) / len(parses) if parses else 0
    avg_fields = sum(len([k for k in schema_keys if k in p]) for p in valid) / len(valid) if valid else 0
    print(f"{model_id:<52}  {rate:>10.0%}    {avg_fields:>18.1f}/{len(schema_keys)}")


Model                                                   Parse rate    Fields present (avg)
────────────────────────────────────────────────────────────────────────────────────────────
gemini-2.5-flash                                            100%                   8.0/8
qwen/qwen3-vl-8b-instruct                                     0%                   0.0/8
qwen/qwen3-vl-30b-a3b-instruct                                0%                   0.0/8
qwen/qwen2.5-vl-72b-instruct                                  0%                   0.0/8
meta-llama/llama-3.2-90b-vision-instruct                      0%                   0.0/8
mistralai/pixtral-large-2411                                  0%                   0.0/8


In [8]:
# ── Field-level agreement vs Gemini baseline ─────────────────────────────────
BASELINE = "gemini-2.5-flash"

if BASELINE not in parsed:
    print(f"Gemini baseline not available — skipping agreement matrix")
else:
    baseline_parses = parsed[BASELINE]
    COMPARE_FIELDS = ["category", "primary_color", "material"]  # most critical fields

    print(f"Agreement with {BASELINE} baseline (per field, across all images):")
    print(f"{'Model':<52}", end="")
    for f in COMPARE_FIELDS:
        print(f"  {f:>14}", end="")
    print()
    print("─" * (52 + len(COMPARE_FIELDS) * 16))

    for model_id, parses in parsed.items():
        if model_id == BASELINE:
            continue
        row = f"{model_id:<52}"
        for field in COMPARE_FIELDS:
            matches = 0
            total = 0
            for i, (b, m) in enumerate(zip(baseline_parses, parses)):
                if b is None or m is None:
                    continue
                total += 1
                bv = str(b.get(field, "")).lower().strip()
                mv = str(m.get(field, "")).lower().strip()
                if bv and mv and bv == mv:
                    matches += 1
            pct = f"{matches/total:.0%}" if total > 0 else "n/a"
            row += f"  {pct:>14}"
        print(row)


Agreement with gemini-2.5-flash baseline (per field, across all images):
Model                                                       category   primary_color        material
────────────────────────────────────────────────────────────────────────────────────────────────────
qwen/qwen3-vl-8b-instruct                                        n/a             n/a             n/a
qwen/qwen3-vl-30b-a3b-instruct                                   n/a             n/a             n/a
qwen/qwen2.5-vl-72b-instruct                                     n/a             n/a             n/a
meta-llama/llama-3.2-90b-vision-instruct                         n/a             n/a             n/a
mistralai/pixtral-large-2411                                     n/a             n/a             n/a


In [9]:
# ── Metrics table ─────────────────────────────────────────────────────────────
rows = []
for model_id, model_results in results.items():
    parses = parsed.get(model_id, [])
    valid_parses = [p for p in parses if p is not None]
    total_cost = sum(r.cost_usd for r in model_results)
    avg_latency = sum(r.latency_s for r in model_results) / len(model_results) if model_results else 0
    cost_actual = all(r.cost_is_actual for r in model_results if r.ok)
    rows.append({
        "Model": model_id.split("/")[-1][:30],
        "Provider": model_results[0].provider if model_results else "",
        "Parse rate": f"{len(valid_parses)/len(parses):.0%}" if parses else "—",
        "Avg latency (s)": f"{avg_latency:.1f}",
        "Total cost (5 imgs)": f"${total_cost:.5f}{'*' if cost_actual else '~'}",
        "Cost per image": f"${total_cost/len(model_results):.6f}" if model_results else "—",
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv(OUT_DIR / "08_metrics.csv", index=False)
print("\n* = actual billed cost   ~ = estimate")
print(f"Saved to {OUT_DIR}/08_metrics.csv")


                        Model   Provider Parse rate Avg latency (s) Total cost (5 imgs) Cost per image
             gemini-2.5-flash     gemini       100%             4.0           $0.00282~      $0.000564
         qwen3-vl-8b-instruct openrouter         0%             0.2           $0.00000*      $0.000000
    qwen3-vl-30b-a3b-instruct openrouter         0%             0.1           $0.00000*      $0.000000
      qwen2.5-vl-72b-instruct openrouter         0%             0.1           $0.00000*      $0.000000
llama-3.2-90b-vision-instruct openrouter         0%             0.1           $0.00000*      $0.000000
           pixtral-large-2411 openrouter         0%             0.1           $0.00000*      $0.000000

* = actual billed cost   ~ = estimate
Saved to ../outputs/08/08_metrics.csv


In [10]:
# ── Side-by-side JSON output grid (first image only) ─────────────────────────
img_idx = 0
model_ids = list(parsed.keys())
n = len(model_ids)

if n == 0:
    print("No results to display")
else:
    fig, axes = plt.subplots(1, n + 1, figsize=((n + 1) * 3.5, 8))
    axes[0].imshow(IMAGES[img_idx])
    axes[0].set_title(f"Input\n{IMAGE_PATHS[img_idx].name[:20]}", fontsize=8)
    axes[0].axis("off")

    for ax, model_id in zip(axes[1:], model_ids):
        p = parsed[model_id][img_idx]
        if p:
            text = json.dumps(p, indent=2)
            # Keep it readable — truncate long descriptions
            lines = text.split("\n")
            short = "\n".join(lines[:20]) + ("\n  ..." if len(lines) > 20 else "")
        else:
            r = results[model_id][img_idx]
            short = f"PARSE FAILED\n\n{r.error or ''}\n\n{r.text[:200] if r.text else ''}"
        ax.text(0.02, 0.98, short, transform=ax.transAxes,
                fontsize=6, verticalalignment="top", fontfamily="monospace",
                bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
        label = model_id.split("/")[-1][:22]
        ax.set_title(label, fontsize=7)
        ax.axis("off")

    plt.suptitle(f"JSON output comparison — image 0", fontsize=11)
    plt.tight_layout(); plt.show()


[image removed from the public copy: third-party or personal photos]


## Decision note

Fill in after reviewing outputs above.

**Winner:** `_______`

**Why:** 
- Parse rate: 
- Category agreement with Gemini baseline: 
- Cost per image vs Gemini: 
- Latency: 

**Ship recommendation for `src/ai/categorize.ts`:**
```
provider: openrouter
model:    _______ 
```

**Open questions:**
- Does the winner handle accessories / non-standard garments as well as Gemini?
- Does adding `existing_tags` / `existing_subcategories` context hurt or help?
